# 🫀 # Project 2:
### Extension 1 Cardiovascular Diseases
## Notebook 2: Model Training and Evaluation

**Author:** Niral Patel
**Dataset:** Cardiovascular Disease Dataset (Kaggle)
**Records:** 68,433 patients × 10 features (after cleaning)
**Target:** cardio — 0=No Disease, 1=Has Disease
**Balance:** 50.6% / 49.4% — NO SMOTE needed

---

In [1]:
# ============================================================
# CELL 2: Imports
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    roc_auc_score, recall_score, f1_score,
    precision_score, accuracy_score,
    classification_report, confusion_matrix,
    roc_curve
)
import joblib, os

plt.style.use('seaborn-v0_8-darkgrid')
print("All imports OK")
print(f"pandas: {pd.__version__}")

All imports OK
pandas: 2.3.3


In [2]:
# ============================================================
# CELL 3: Load Final Cleaned Dataset
# ============================================================

# Load clean engineered dataset
# This is the df saved at end of EDA notebook Cell 11
df = pd.read_csv('../notebook/data/cardio_train.csv', sep=';')

# Re-apply all cleaning steps (must match EDA notebook)
df['age']  = (df['age'] / 365).round()
df = df.drop(columns=['id'])

# Re-apply cleaning
df = df[
    (df['ap_hi'] >= 70)  & (df['ap_hi'] <= 200) &
    (df['ap_lo'] >= 40)  & (df['ap_lo'] <= 130) &
    (df['height'] >= 140) & (df['height'] <= 220) &
    (df['weight'] >= 30)  & (df['weight'] <= 200) &
    (df['ap_hi'] > df['ap_lo'])
].copy()

# Re-apply feature engineering
df['BMI']            = (df['weight'] /
                        (df['height']/100)**2).round(1)
df['pulse_pressure'] = df['ap_hi'] - df['ap_lo']

def bp_cat(row):
    s, d = row['ap_hi'], row['ap_lo']
    if   s < 120 and d < 80:  return 1
    elif s < 130 and d < 80:  return 2
    elif s < 140 or  d < 90:  return 3
    elif s < 180 or  d < 120: return 4
    else:                      return 5

df['bp_category'] = df.apply(bp_cat, axis=1)

# Remove extreme BMI
df = df[(df['height'] >= 140) & (df['BMI'] <= 60)].copy()

# Drop weak + redundant features
drop_cols = ['age_group', 'BMI_Category',
             'gender', 'alco', 'smoke', 'height']
# Only drop if they exist
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

# Separate X and y
X = df.drop(columns=['cardio'])
y = df['cardio']

print(f"X shape: {X.shape}")
print(f"y balance: {y.value_counts().to_dict()}")
print(f"Features: {list(X.columns)}")

X shape: (68433, 10)
y balance: {0: 34595, 1: 33838}
Features: ['age', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'active', 'BMI', 'pulse_pressure', 'bp_category']


In [6]:
# ============================================================
# CELL 4: Train/Test Split + Scale + Train 4 Models
# Note: NO SMOTE — dataset is balanced 50/50
# ============================================================

# ── Step 1: Split ──────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size   = 0.2,
    random_state= 42,
    stratify    = y
)

print("=" * 55)
print(" TRAIN / TEST SPLIT")
print("=" * 55)
print(f"\n X_train : {X_train.shape[0]:,} × {X_train.shape[1]}")
print(f" X_test  : {X_test.shape[0]:,} × {X_test.shape[1]}")
print(f"\n Train balance: {y_train.value_counts().to_dict()}")
print(f" Test  balance: {y_test.value_counts().to_dict()}")

# ── Step 2: Scale ──────────────────────────────────────────
# 🔒 fit on TRAIN only — transform both
scaler       = StandardScaler()
X_train_sc   = scaler.fit_transform(X_train)
X_test_sc    = scaler.transform(X_test)

print(f"\n StandardScaler fitted on train ✓")

# ── Step 3: Define 4 models ────────────────────────────────
# 💡 No class_weight or scale_pos_weight needed — balanced!
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42),
    'Random Forest'      : RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost'            : XGBClassifier(
        n_estimators=100, random_state=42,
        eval_metric='logloss', verbosity=0),
    'LightGBM'           : LGBMClassifier(
        n_estimators=100, random_state=42, verbose=-1)
}

# ── Step 4: Train + Evaluate ───────────────────────────────
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    return {
        'Model'    : name,
        'AUC'      : round(roc_auc_score(y_te, y_prob), 4),
        'Recall'   : round(recall_score(y_te, y_pred), 4),
        'Precision': round(precision_score(y_te, y_pred), 4),
        'F1'       : round(f1_score(y_te, y_pred), 4),
        'Accuracy' : round(accuracy_score(y_te, y_pred), 4),
        '_prob'    : y_prob,
        '_pred'    : y_pred,
        '_model'   : model
    }

print(f"\n{'=' * 55}")
print(" TRAINING 4 MODELS")
print(f"{'=' * 55}")

results = []
for name, model in models.items():
    print(f"\n Training {name}...", end=' ')
    # Tree models don't need scaling
    # LR needs scaling
    X_tr = X_train_sc if name == 'Logistic Regression' \
           else X_train.values
    X_te = X_test_sc  if name == 'Logistic Regression' \
           else X_test.values
    r = evaluate(name, model, X_tr, y_train, X_te, y_test)
    results.append(r)
    print(f"AUC={r['AUC']:.4f}  "
          f"Recall={r['Recall']:.4f}  "
          f"F1={r['F1']:.4f}")

# ── Step 5: Comparison table ───────────────────────────────
print(f"\n{'=' * 65}")
print(" BASELINE MODEL COMPARISON — BALANCED DATA (no SMOTE)")
print(f"{'=' * 65}")
res_df = pd.DataFrame([
    {k: v for k, v in r.items() if not k.startswith('_')}
    for r in results
]).sort_values('AUC', ascending=False)
print(res_df.to_string(index=False))

print(f"\n Winner: {res_df.iloc[0]['Model']}")
print(f"  AUC  : {res_df.iloc[0]['AUC']:.4f}")
print(f"  F1   : {res_df.iloc[0]['F1']:.4f}")

 TRAIN / TEST SPLIT

 X_train : 54,746 × 10
 X_test  : 13,687 × 10

 Train balance: {0: 27676, 1: 27070}
 Test  balance: {0: 6919, 1: 6768}

 StandardScaler fitted on train ✓

 TRAINING 4 MODELS

 Training Logistic Regression... AUC=0.7867  Recall=0.6603  F1=0.7029

 Training Random Forest... AUC=0.7493  Recall=0.6842  F1=0.6877

 Training XGBoost... AUC=0.7886  Recall=0.6865  F1=0.7138

 Training LightGBM... AUC=0.7942  Recall=0.6894  F1=0.7171

 BASELINE MODEL COMPARISON — BALANCED DATA (no SMOTE)
              Model    AUC  Recall  Precision     F1  Accuracy
           LightGBM 0.7942  0.6894     0.7470 0.7171    0.7310
            XGBoost 0.7886  0.6865     0.7435 0.7138    0.7278
Logistic Regression 0.7867  0.6603     0.7515 0.7029    0.7240
      Random Forest 0.7493  0.6842     0.6912 0.6877    0.6927

 Winner: LightGBM
  AUC  : 0.7942
  F1   : 0.7171


---
## 📊 Cell 4 — Baseline Model Comparison

### Key Insight: Balanced Data Benefits ALL Models

| Model | AUC | F1 | Recall | Precision |
|-------|-----|-----|--------|-----------|
| **LightGBM** | **0.7942** | **0.7171** | 0.6894 | 0.7470 |
| XGBoost | 0.7886 | 0.7138 | 0.6865 | 0.7435 |
| Logistic Regression | 0.7867 | 0.7029 | 0.6603 | 0.7515 |
| Random Forest | 0.7493 | 0.6877 | 0.6842 | 0.6912 |

### vs Project 2 (Diabetes, imbalanced)
- Project 2 best F1 = **0.44** (required SMOTE + heavy tuning)
- Project 3 best F1 = **0.72** with zero preprocessing tricks
- Lesson: balanced data = all models competitive from baseline

### Clinical Interpretation
- Precision (~0.74) > Recall (~0.69) — model is slightly conservative
- Of 100 patients flagged as having heart disease → 74 actually do
- Of 100 patients with actual heart disease → model catches 69

### Winner: LightGBM
- Fastest training time
- Best AUC and F1 at baseline
- Will tune with GridSearchCV next
---

In [8]:
# ============================================================
# CELL 5: Hyperparameter Tuning — LightGBM
# ============================================================
from sklearn.model_selection import GridSearchCV, StratifiedKFold

print("=" * 60)
print(" HYPERPARAMETER TUNING — LightGBM (Winner)")
print(" Balanced data — no class_weight needed")
print("=" * 60)

param_grid = {
    'n_estimators'  : [100, 200, 300],
    'learning_rate' : [0.01, 0.05, 0.1],
    'max_depth'     : [3, 5, 7],
    'num_leaves'    : [20, 31, 50],
    'subsample'     : [0.7, 0.8, 1.0],
}

total = 3*3*3*3*3
print(f"\n Combinations : {total}")
print(f" CV folds     : 5")
print(f" Total fits   : {total*5}")
print(f"\n Using RandomizedSearchCV (n_iter=40)...")

from sklearn.model_selection import RandomizedSearchCV

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator  = LGBMClassifier(random_state=42, verbose=-1),
    param_distributions = param_grid,
    n_iter     = 40,
    scoring    = 'roc_auc',
    cv         = cv,
    n_jobs     = -1,
    verbose    = 0,
    random_state = 42,
    refit      = True
)

# Use scaled data for consistency
search.fit(X_train_sc, y_train)

best_lgb  = search.best_estimator_
y_pred_t  = best_lgb.predict(X_test_sc)
y_prob_t  = best_lgb.predict_proba(X_test_sc)[:, 1]

print(f"\n{'=' * 60}")
print(" TUNING RESULTS")
print(f"{'=' * 60}")
print(f"\n Best params  : {search.best_params_}")
print(f" Best CV AUC  : {search.best_score_:.4f}")
print(f"\n Before tuning: AUC=0.7942  F1=0.7171")
print(f" After tuning :")
print(f"   AUC       : {roc_auc_score(y_test, y_prob_t):.4f}")
print(f"   Recall    : {recall_score(y_test, y_pred_t):.4f}")
print(f"   F1        : {f1_score(y_test, y_pred_t):.4f}")
print(f"   Precision : {precision_score(y_test, y_pred_t):.4f}")
print(f"   Accuracy  : {accuracy_score(y_test, y_pred_t):.4f}")

print(f"\n Classification Report:")
print(classification_report(
    y_test, y_pred_t,
    target_names=['No Disease', 'Disease']
))

 HYPERPARAMETER TUNING — LightGBM (Winner)
 Balanced data — no class_weight needed

 Combinations : 243
 CV folds     : 5
 Total fits   : 1215

 Using RandomizedSearchCV (n_iter=40)...

 TUNING RESULTS

 Best params  : {'subsample': 0.7, 'num_leaves': 20, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.05}
 Best CV AUC  : 0.8019

 Before tuning: AUC=0.7942  F1=0.7171
 After tuning :
   AUC       : 0.7948
   Recall    : 0.6828
   F1        : 0.7164
   Precision : 0.7535
   Accuracy  : 0.7327

 Classification Report:
              precision    recall  f1-score   support

  No Disease       0.72      0.78      0.75      6919
     Disease       0.75      0.68      0.72      6768

    accuracy                           0.73     13687
   macro avg       0.73      0.73      0.73     13687
weighted avg       0.73      0.73      0.73     13687



In [9]:
# ============================================================
# CELL 6: Interaction Features + CatBoost
# ============================================================
from catboost import CatBoostClassifier

# ── Part 1: Create Interaction Features ───────────────────
print("=" * 60)
print(" PART 1: INTERACTION FEATURES")
print("=" * 60)

# Make a copy for feature engineering
X_eng = X.copy()

# 1. Age × Systolic BP — age-weighted hypertension risk
#    Older patient with high BP = exponentially more dangerous
X_eng['age_x_aphi']  = X_eng['age'] * X_eng['ap_hi']

# 2. BMI × Cholesterol — metabolic syndrome composite
#    Obese + high cholesterol = classic heart disease profile
X_eng['bmi_x_chol']  = X_eng['BMI'] * X_eng['cholesterol']

# 3. Pulse Pressure × Age — arterial stiffness with age
#    High pulse pressure in elderly = hardened arteries
X_eng['pp_x_age']    = X_eng['pulse_pressure'] * X_eng['age']

print(f"\n Original features : {X.shape[1]}")
print(f" New features      : 3 interaction terms")
print(f" Total features    : {X_eng.shape[1]}")
print(f"\n New feature stats:")
print(X_eng[['age_x_aphi','bmi_x_chol','pp_x_age']
            ].describe().round(2).T[['mean','std','min','max']])

# ── Correlation of new features with target ────────────────
df_check = X_eng.copy()
df_check['cardio'] = y.values
new_corr = df_check[['age_x_aphi','bmi_x_chol',
                      'pp_x_age','cardio']].corr()['cardio']
print(f"\n Correlation with target:")
print(f"  age_x_aphi : {new_corr['age_x_aphi']:.4f}")
print(f"  bmi_x_chol : {new_corr['bmi_x_chol']:.4f}")
print(f"  pp_x_age   : {new_corr['pp_x_age']:.4f}")
print(f"\n  (Original ap_hi was : +0.431)")
print(f"  (Original age was   : +0.239)")

# ── Part 2: Retrain LightGBM with new features ─────────────
print(f"\n{'=' * 60}")
print(" PART 2: LIGHTGBM WITH INTERACTION FEATURES")
print("=" * 60)

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_eng, y, test_size=0.2,
    random_state=42, stratify=y
)

scaler2     = StandardScaler()
X_tr2_sc    = scaler2.fit_transform(X_tr2)
X_te2_sc    = scaler2.transform(X_te2)

# Reuse best params from tuning
best_lgb_eng = LGBMClassifier(
    n_estimators=100, learning_rate=0.05,
    max_depth=5, num_leaves=20,
    subsample=0.7, random_state=42, verbose=-1
)
best_lgb_eng.fit(X_tr2_sc, y_tr2)
y_prob_eng = best_lgb_eng.predict_proba(X_te2_sc)[:, 1]
y_pred_eng = best_lgb_eng.predict(X_te2_sc)

auc_eng = roc_auc_score(y_te2, y_prob_eng)
f1_eng  = f1_score(y_te2, y_pred_eng)
print(f"\n LightGBM + interaction features:")
print(f"  AUC    : {auc_eng:.4f}  (was 0.7948)")
print(f"  F1     : {f1_eng:.4f}  (was 0.7164)")
print(f"  Change : {auc_eng-0.7948:+.4f} AUC")

# ── Part 3: CatBoost ───────────────────────────────────────
print(f"\n{'=' * 60}")
print(" PART 3: CatBoost (new model)")
print("=" * 60)

# CatBoost on ORIGINAL features (no scaling needed)
X_tr_cat, X_te_cat, y_tr_cat, y_te_cat = train_test_split(
    X, y, test_size=0.2,
    random_state=42, stratify=y
)

# CatBoost with interaction features
X_tr_cat2, X_te_cat2, _, _ = train_test_split(
    X_eng, y, test_size=0.2,
    random_state=42, stratify=y
)

print("\n Training CatBoost (default params — no tuning)...")
cat_model = CatBoostClassifier(
    iterations  = 200,
    learning_rate = 0.05,
    depth       = 6,
    random_seed = 42,
    verbose     = 0,          # silent
    eval_metric = 'AUC'
)
# CatBoost does NOT need scaling
cat_model.fit(X_tr_cat, y_tr_cat)
y_prob_cat = cat_model.predict_proba(X_te_cat)[:, 1]
y_pred_cat = cat_model.predict(X_te_cat)

auc_cat = roc_auc_score(y_te_cat, y_prob_cat)
f1_cat  = f1_score(y_te_cat, y_pred_cat)
print(f"\n CatBoost (original features, no tuning):")
print(f"  AUC    : {auc_cat:.4f}")
print(f"  F1     : {f1_cat:.4f}")

# CatBoost with interaction features
print("\n Training CatBoost + interaction features...")
cat_eng = CatBoostClassifier(
    iterations=200, learning_rate=0.05,
    depth=6, random_seed=42, verbose=0,
    eval_metric='AUC'
)
cat_eng.fit(X_tr_cat2, y_tr_cat)
y_prob_cat2 = cat_eng.predict_proba(X_te_cat2)[:, 1]
y_pred_cat2 = cat_eng.predict(X_te_cat2)

auc_cat2 = roc_auc_score(y_te_cat, y_prob_cat2)
f1_cat2  = f1_score(y_te_cat, y_pred_cat2)
print(f"\n CatBoost + interaction features:")
print(f"  AUC    : {auc_cat2:.4f}")
print(f"  F1     : {f1_cat2:.4f}")

# ── Final Comparison ───────────────────────────────────────
print(f"\n{'=' * 65}")
print(" FINAL COMPARISON — ALL EXPERIMENTS")
print(f"{'=' * 65}")
print(f"\n  {'Model':<40} {'AUC':>8} {'F1':>8}")
print(f"  {'-'*58}")
print(f"  {'LightGBM baseline (no tuning)':<40} {'0.7942':>8} {'0.7171':>8}")
print(f"  {'LightGBM tuned':<40} {'0.7948':>8} {'0.7164':>8}")
print(f"  {'LightGBM + interaction features':<40} {auc_eng:>8.4f} {f1_eng:>8.4f}")
print(f"  {'CatBoost (no tuning)':<40} {auc_cat:>8.4f} {f1_cat:>8.4f}")
print(f"  {'CatBoost + interaction features':<40} {auc_cat2:>8.4f} {f1_cat2:>8.4f}")

all_aucs = {
    'LightGBM+interactions': auc_eng,
    'CatBoost baseline'    : auc_cat,
    'CatBoost+interactions': auc_cat2
}
winner = max(all_aucs, key=all_aucs.get)
print(f"\n  WINNER: {winner} → AUC={all_aucs[winner]:.4f}")

 PART 1: INTERACTION FEATURES

 Original features : 10
 New features      : 3 interaction terms
 Total features    : 13

 New feature stats:
               mean      std     min      max
age_x_aphi  6774.16  1352.96  2800.0  12800.0
bmi_x_chol    38.03    22.37    11.7    176.4
pp_x_age    2430.54   745.45   290.0   8320.0

 Correlation with target:
  age_x_aphi : 0.4258
  bmi_x_chol : 0.2513
  pp_x_age   : 0.3751

  (Original ap_hi was : +0.431)
  (Original age was   : +0.239)

 PART 2: LIGHTGBM WITH INTERACTION FEATURES

 LightGBM + interaction features:
  AUC    : 0.7953  (was 0.7948)
  F1     : 0.7156  (was 0.7164)
  Change : +0.0005 AUC

 PART 3: CatBoost (new model)

 Training CatBoost (default params — no tuning)...

 CatBoost (original features, no tuning):
  AUC    : 0.7958
  F1     : 0.7172

 Training CatBoost + interaction features...

 CatBoost + interaction features:
  AUC    : 0.7956
  F1     : 0.7174

 FINAL COMPARISON — ALL EXPERIMENTS

  Model                          